# Etapa 3 – Processamento da base diária

Este notebook lê os arquivos brutos baixados por `01_coleta.ipynb` e monta a base usada na análise:

- `dados/processados/base_diaria.csv`: uma linha por dia útil comum a Brasil e EUA, com níveis, retornos, VIX, drawdowns e indicadores de regime de crise;
- `dados/processados/log_processamento.md`: origem dos dados, dias excluídos, resultado das verificações e contagem de dias por regime;
- `dados/dicionario.md`: descrição de cada coluna da base.

O passo a passo e as justificativas estão em `codigo/plano-processamento.md`. As decisões de fundo estão no `plano-de-trabalho.md`, seção 3.1.

**Regra central:** os retornos são calculados **depois** de alinhar os níveis no calendário comum. Assim, quando um dia é excluído (feriado em um dos países), o retorno do dia seguinte acumula os dois dias para todas as séries ao mesmo tempo, e nenhum movimento se perde.

## 1. Configuração

Parâmetros da amostra e das definições de crise. Mudar um limiar aqui e rodar o notebook de novo refaz toda a base.

In [11]:
import hashlib
import json
import warnings
import zipfile
import xml.etree.ElementTree as ET
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# Os arquivos da ANBIMA não têm estilo padrão; o aviso do openpyxl é inofensivo.
warnings.filterwarnings("ignore", message="Workbook contains no default style")

RAIZ = Path.cwd().parent if Path.cwd().name == "codigo" else Path.cwd()

# Pasta da coleta usada. None = coleta mais recente com manifesto.
PASTA_BRUTOS = None
if PASTA_BRUTOS is None:
    PASTA_BRUTOS = sorted(p.parent for p in (RAIZ / "dados" / "brutos").glob("*/manifesto.csv"))[-1]
PASTA_BRUTOS = Path(PASTA_BRUTOS)
PASTA_SAIDA = RAIZ / "dados" / "processados"

# Amostra: do 1º valor do IMA-B até a data de corte fixa (decisão da seção 7 do plano de processamento).
INICIO_AMOSTRA = pd.Timestamp("2003-09-16")
FIM_AMOSTRA = pd.Timestamp("2026-09-30")

# Crises
LIMIAR_VIX = 30               # crise global (principal)
PERCENTIL_VIX = 0.90          # crise global (robustez)
LIMIAR_DRAWDOWN = -0.20       # crise local / crise EUA
JANELA_DRAWDOWN = 252         # pregões
PREGOES_POS_GLOBAL = 63       # robustez (a): exclui de so_local os dias logo após crise global
JANELA_CHOQUE = 21            # robustez (c): retorno acumulado do Ibovespa em 21 pregões
PERCENTIL_CHOQUE = 0.05
PREGOES_EVENTO = 21           # robustez (b): janela após cada evento da lista fixa

# Lista fixa de crises locais (robustez b). PRELIMINAR: validar datas e eventos com a literatura (etapa 2).
EVENTOS_LOCAIS = {
    "2015-09-09": "S&P retira o grau de investimento do Brasil",
    "2015-12-02": "Abertura do processo de impeachment na Câmara",
    "2017-05-18": "Joesley Day (divulgação da gravação na noite de 17/05)",
    "2018-05-21": "Início da greve dos caminhoneiros",
}

# Treasury: título de 10 anos ao par, cupom semestral, reprecificado todo dia.
MATURIDADE_ANOS = 10

# Limpeza do BRL=X: descarta observações que se afastam mais que isto da PTAX do mesmo dia.
LIMIAR_DESVIO_BRL = 0.05

print("Brutos:", PASTA_BRUTOS.relative_to(RAIZ))
print("Amostra:", INICIO_AMOSTRA.date(), "a", FIM_AMOSTRA.date())

Brutos: dados/brutos/2026-10-21
Amostra: 2003-09-16 a 2026-09-30


## 2. Leitura e padronização

Cada função devolve uma série com índice de data (sem hora) e nome padronizado. Nenhum valor é alterado aqui, exceto onde indicado:

- **H.15:** observações sem dado vêm com `OBS_STATUS="ND"` e valor −9999; viram ausentes.
- **ANBIMA:** usa só o `Número Índice` e a `Duration (d.u.)`. A coluna `Variação Diária (%)` tem erros de digitação e não é usada no cálculo; ela é lida só para a verificação da seção 7. Duration igual a 0 vira ausente.
- **SGS:** o valor vem como texto e a data como dd/mm/aaaa.
- **Yahoo:** usa `Close` nos índices (para eles, `Adj Close` = `Close`) e `Adj Close` no IEF (inclui dividendos).

In [12]:
def ler_yahoo(nome, coluna="Close"):
    dados = pd.read_csv(PASTA_BRUTOS / f"yahoo_{nome}.csv", parse_dates=["Date"], index_col="Date")
    return dados[coluna].rename(nome).sort_index()


def ler_anbima(codigo):
    dados = pd.read_excel(PASTA_BRUTOS / f"anbima_{codigo}.xlsx")
    dados = dados.set_index("Data de Referência").sort_index()
    dados.index.name = None
    return dados


def ler_sgs(codigo):
    registros = json.loads((PASTA_BRUTOS / f"sgs_{codigo}.json").read_text())
    dados = pd.DataFrame(registros)
    serie = pd.Series(dados["valor"].astype(float).values,
                      index=pd.to_datetime(dados["data"], format="%d/%m/%Y"))
    return serie.sort_index()


def ler_h15(series):
    """Extrai séries diárias do XML (SDMX) do pacote completo do H.15."""
    obs = {s: {} for s in series}
    atual = None
    with zipfile.ZipFile(PASTA_BRUTOS / "h15_completo.zip") as z:
        for evento, el in ET.iterparse(z.open("H15_data.xml"), events=("start", "end")):
            tag = el.tag.rsplit("}", 1)[-1]
            if evento == "start" and tag == "Series":
                atual = el.get("SERIES_NAME") if el.get("SERIES_NAME") in obs else None
            elif evento == "end" and tag == "Obs" and atual:
                valor = float(el.get("OBS_VALUE")) if el.get("OBS_STATUS") == "A" else np.nan
                obs[atual][pd.Timestamp(el.get("TIME_PERIOD"))] = valor
            elif evento == "end" and tag == "Series":
                el.clear()
    return {s: pd.Series(v).sort_index() for s, v in obs.items()}


def ler_cboe_vix():
    dados = pd.read_csv(PASTA_BRUTOS / "cboe_vix.csv")
    dados.index = pd.to_datetime(dados["DATE"], format="%m/%d/%Y")
    return dados["CLOSE"].sort_index()


sp500 = ler_yahoo("gspc").rename("sp500")
ibov = ler_yahoo("bvsp").rename("ibov")
ief = ler_yahoo("ief", "Adj Close").rename("ief")
usdbrl_fech_bruto = ler_yahoo("brlx").rename("usdbrl_fech")

anbima = {codigo: ler_anbima(codigo) for codigo in ["IMAB", "IMAB5MAIS", "IDKAIPCA10A", "IRFM"]}
nomes_anbima = {"IMAB": "imab", "IMAB5MAIS": "imab5mais", "IDKAIPCA10A": "idka10", "IRFM": "irfm"}
niveis_anbima = {nomes_anbima[c]: d["Número Índice"].rename(nomes_anbima[c]) for c, d in anbima.items()}
duration_anbima = {
    f"{nomes_anbima[c]}_duration_anos": (d["Duration (d.u.)"].replace(0, np.nan) / 252).rename(f"{nomes_anbima[c]}_duration_anos")
    for c, d in anbima.items() if "Duration (d.u.)" in d
}

usdbrl_ptax = ler_sgs(1).rename("usdbrl_ptax")

h15 = ler_h15(["RIFLGFCY10_N.B", "RIFLGFCY10_XII_N.B"])
y_treasury10 = h15["RIFLGFCY10_N.B"].rename("y_treasury10")
y_tips10 = h15["RIFLGFCY10_XII_N.B"].rename("y_tips10")
vix = ler_cboe_vix().rename("vix")

for s in [sp500, ibov, ief, usdbrl_fech_bruto, *niveis_anbima.values(), usdbrl_ptax, y_treasury10, y_tips10, vix]:
    print(f"{s.name:<14} {s.index.min().date()} a {s.index.max().date()}  {s.notna().sum():>6} obs. válidas")

sp500          2002-09-03 a 2026-10-20    6072 obs. válidas
ibov           2002-09-02 a 2026-10-20    5982 obs. válidas
ief            2002-09-03 a 2026-10-20    6072 obs. válidas
usdbrl_fech    2003-12-01 a 2026-10-20    5520 obs. válidas
imab           2003-09-16 a 2026-10-21    5802 obs. válidas
imab5mais      2003-09-16 a 2026-10-21    5802 obs. válidas
idka10         2005-12-30 a 2026-10-21    5224 obs. válidas
irfm           2000-02-01 a 2026-10-21    6707 obs. válidas
usdbrl_ptax    2002-09-02 a 2026-10-21    6064 obs. válidas
y_treasury10   1962-01-02 a 2026-10-20   16185 obs. válidas
y_tips10       2003-01-02 a 2026-10-20    5954 obs. válidas
vix            1990-01-02 a 2026-10-20    9298 obs. válidas


## 3. Limpeza do câmbio de fechamento (BRL=X)

O `BRL=X` do Yahoo tem saltos isolados que se revertem no dia seguinte (por exemplo, 08/05/2006, 27/01/2012 e 11/05/2020). A regra é descartar as observações que se afastam mais de 5% da PTAX do mesmo dia. A PTAX é medida entre 10h e 13h, então diferenças menores são esperadas em dias voláteis. O `BRL=X` é só robustez da P3; a PTAX é a série principal.

In [13]:
desvio_brl = (usdbrl_fech_bruto / usdbrl_ptax.reindex(usdbrl_fech_bruto.index) - 1)
descartes_brl = desvio_brl[desvio_brl.abs() > LIMIAR_DESVIO_BRL]
usdbrl_fech = usdbrl_fech_bruto.mask(desvio_brl.abs() > LIMIAR_DESVIO_BRL)
print(f"Observações descartadas do BRL=X: {len(descartes_brl)}")
print(pd.DataFrame({"BRL=X": usdbrl_fech_bruto[descartes_brl.index],
                    "PTAX": usdbrl_ptax.reindex(descartes_brl.index),
                    "desvio": descartes_brl.round(4)}))

Observações descartadas do BRL=X: 6
             BRL=X    PTAX  desvio
Date                              
2006-05-08  1.9472  2.0651 -0.0571
2006-07-14  2.0968  2.2130 -0.0525
2006-08-16  2.0144  2.1329 -0.0556
2012-01-27  1.8525  1.7436  0.0625
2017-05-18  3.1334  3.3807 -0.0732
2020-05-11  5.4770  5.7954 -0.0549


## 4. Drawdowns e choques no calendário de cada bolsa

Os drawdowns são calculados **antes do alinhamento**, no calendário próprio de cada bolsa, para que a janela de 252 dias corresponda a 252 pregões de verdade. A coleta começa em set. 2002, então a janela já está completa no início da amostra.

- `dd_*` = nível / máximo dos últimos 252 pregões − 1;
- `r21_ibov` = log-retorno acumulado do Ibovespa nos últimos 21 pregões (robustez c).

In [14]:
def drawdown(nivel, janela=JANELA_DRAWDOWN):
    return nivel / nivel.rolling(janela, min_periods=janela).max() - 1


dd_ibov = drawdown(ibov).rename("dd_ibov")
dd_sp500 = drawdown(sp500).rename("dd_sp500")
r21_ibov = (100 * np.log(ibov / ibov.shift(JANELA_CHOQUE))).rename("r21_ibov")

print("Primeiro drawdown válido: Ibovespa", dd_ibov.first_valid_index().date(),
      "| S&P 500", dd_sp500.first_valid_index().date())

Primeiro drawdown válido: Ibovespa 2003-09-04 | S&P 500 2003-09-02


## 5. Alinhamento no calendário comum

O calendário comum são os dias em que **todas as séries principais** têm valor: S&P 500, Treasury 10 anos, VIX, Ibovespa, IMA-B e PTAX. As séries de robustez entram nesse calendário sem reduzi-lo (ficam vazias onde não existem).

A célula também conta, dentro da amostra, as datas em que alguma série principal faltava e quais faltavam (inclui feriados comuns aos dois países).

In [15]:
principais = pd.concat([sp500, y_treasury10, vix, ibov, niveis_anbima["imab"], usdbrl_ptax], axis=1, sort=True)
robustez = pd.concat([niveis_anbima["imab5mais"], niveis_anbima["idka10"], niveis_anbima["irfm"],
                      ief, y_tips10, usdbrl_fech, *duration_anbima.values()], axis=1, sort=True)
estado = pd.concat([dd_ibov, dd_sp500, r21_ibov], axis=1, sort=True)

calendario = principais.dropna().index
base = pd.concat([principais.reindex(calendario), robustez.reindex(calendario),
                  estado.reindex(calendario)], axis=1)

# Dias excluídos dentro da amostra, por série principal ausente
na_amostra = principais.loc[INICIO_AMOSTRA:FIM_AMOSTRA]
excluidos = na_amostra[na_amostra.isna().any(axis=1)]
motivos = excluidos.isna().apply(lambda linha: ", ".join(linha.index[linha]), axis=1)
tabela_excluidos = motivos.value_counts().rename("dias").to_frame()
tabela_excluidos.index.name = "séries ausentes"
print(f"Dias no calendário comum (amostra): {len(base.loc[INICIO_AMOSTRA:FIM_AMOSTRA])}")
print(f"Datas com alguma série principal ausente (amostra): {len(excluidos)}")
tabela_excluidos.head(12)

Dias no calendário comum (amostra): 5526
Datas com alguma série principal ausente (amostra): 486


,dias
séries ausentes,
"ibov, imab, usdbrl_ptax",154
"sp500, y_treasury10, vix",112
ibov,74
"sp500, y_treasury10, vix, ibov, imab, usdbrl_ptax",56
y_treasury10,40
"sp500, y_treasury10",29
"sp500, vix, ibov, imab, usdbrl_ptax",7
"sp500, y_treasury10, vix, ibov",4
"sp500, y_treasury10, ibov, imab, usdbrl_ptax",4


## 6. Retornos

Log-retornos em % (100 · ln(P_t / P_t−1)) calculados sobre os níveis já alinhados.

**Treasury e TIPS:** o H.15 publica a taxa, não o preço. O retorno diário é o de um título de 10 anos **ao par**, com cupom semestral igual à taxa do dia anterior, reavaliado à taxa do dia com o prazo reduzido pelo tempo decorrido (τ = dias corridos / 365), mais o cupom corrido (*carry*). Assim entram *carry*, duration e convexidade, sem precisar fixar uma duration. A aproximação −D·Δy (com D modificada do título ao par e o *carry*) fica na base como conferência (`r_treasury_aprox`).

**Retornos em dólar:** `r_ibov_usd` = `r_ibov` − `r_usdbrl` (o câmbio está em R$/US$).

In [16]:
def log_retorno(nivel):
    return 100 * np.log(nivel / nivel.shift(1))


def preco_par(c, y, n):
    """Preço (base 100) de um título com cupom anual c, pago semestralmente, n anos até o vencimento, à taxa y."""
    desconto = (1 + y / 2) ** (-2 * n)
    with np.errstate(divide="ignore", invalid="ignore"):
        anuidade = np.where(np.abs(y) < 1e-10, c * n, c / y * (1 - desconto))
    return 100 * (anuidade + desconto)


def retorno_titulo_par(yield_pct, datas, maturidade=MATURIDADE_ANOS):
    y = yield_pct.values / 100
    tau = np.r_[np.nan, np.diff(datas.values).astype("timedelta64[D]").astype(float)] / 365
    c = np.r_[np.nan, y[:-1]]
    preco = preco_par(c, y, maturidade - tau)
    retorno = (preco + 100 * c * tau) / 100 - 1
    return pd.Series(100 * np.log1p(retorno), index=datas)


def retorno_aprox(yield_pct, datas, maturidade=MATURIDADE_ANOS):
    y_ant = yield_pct.shift(1) / 100
    duration = (1 - (1 + y_ant / 2) ** (-2 * maturidade)) / y_ant
    tau = pd.Series(datas, index=datas).diff().dt.days / 365
    return 100 * (y_ant * tau - duration * yield_pct.diff() / 100)


for nome in ["sp500", "ibov", "imab", "imab5mais", "idka10", "irfm", "ief", "usdbrl_ptax", "usdbrl_fech"]:
    base[f"r_{nome}"] = log_retorno(base[nome])
base = base.rename(columns={"r_imab": "r_ntnb", "r_imab5mais": "r_ntnb5mais", "r_usdbrl_ptax": "r_usdbrl"})

base["r_treasury"] = retorno_titulo_par(base["y_treasury10"], base.index)
base["r_treasury_aprox"] = retorno_aprox(base["y_treasury10"], base.index)
base["r_tips"] = retorno_titulo_par(base["y_tips10"], base.index)

base["r_ibov_usd"] = base["r_ibov"] - base["r_usdbrl"]
base["r_ntnb_usd"] = base["r_ntnb"] - base["r_usdbrl"]
base["dias_corridos"] = pd.Series(base.index, index=base.index).diff().dt.days

base.loc[INICIO_AMOSTRA:FIM_AMOSTRA, ["r_sp500", "r_treasury", "r_treasury_aprox", "r_ief", "r_ibov", "r_ntnb", "r_usdbrl"]].describe().round(3)

,r_sp500,r_treasury,r_treasury_aprox,r_ief,r_ibov,r_ntnb,r_usdbrl
count,5525.000,5525.000,5525.000,5525.000,5525.000,5525.000,5525.000
mean,0.036,0.011,0.011,0.013,0.044,0.051,0.011
std,1.191,0.490,0.490,0.432,1.657,0.411,0.930
min,-12.765,-3.114,-3.123,-2.551,-15.993,-7.818,-9.584
25%,-0.414,-0.272,-0.273,-0.245,-0.793,-0.093,-0.476
50%,0.074,0.011,0.011,0.018,0.061,0.060,-0.014
75%,0.574,0.281,0.281,0.263,0.945,0.213,0.448
max,10.424,4.395,4.382,3.369,15.473,5.260,8.917


## 7. Variáveis de regime de crise

| Variável | Definição |
|---|---|
| `crise_global` | VIX > 30 |
| `crise_global_p90` | VIX acima do percentil 90 da amostra (robustez) |
| `crise_local_br` | drawdown do Ibovespa ≤ −20% |
| `crise_eua` | drawdown do S&P 500 ≤ −20% |
| `crise_p1_eua` | `crise_global` ou `crise_eua` (crises de aversão a risco nos EUA, P1) |
| `so_local`, `so_global`, `ambas`, `normal` | combinações de `crise_local_br` e `crise_global` (P4) |
| `so_local_p90` | idem, com `crise_global_p90` |
| `so_local_sem_cauda` | `so_local` sem os 63 pregões seguintes a um dia de `crise_global` (robustez a) |
| `crise_local_fixa` | 21 pregões a partir de cada evento da lista fixa (robustez b) |
| `choque_local_br` | retorno do Ibovespa em 21 pregões abaixo do percentil 5 da amostra (robustez c) |
| `so_local_choque` | `choque_local_br` e não `crise_global` |

Os percentis são calculados sobre a amostra (de `INICIO_AMOSTRA` a `FIM_AMOSTRA`).

In [17]:
amostra = base.loc[INICIO_AMOSTRA:FIM_AMOSTRA]
limiar_vix_p90 = amostra["vix"].quantile(PERCENTIL_VIX)
limiar_choque = amostra["r21_ibov"].quantile(PERCENTIL_CHOQUE)

base["crise_global"] = base["vix"] > LIMIAR_VIX
base["crise_global_p90"] = base["vix"] > limiar_vix_p90
base["crise_local_br"] = base["dd_ibov"] <= LIMIAR_DRAWDOWN
base["crise_eua"] = base["dd_sp500"] <= LIMIAR_DRAWDOWN
base["crise_p1_eua"] = base["crise_global"] | base["crise_eua"]

base["so_local"] = base["crise_local_br"] & ~base["crise_global"]
base["so_global"] = base["crise_global"] & ~base["crise_local_br"]
base["ambas"] = base["crise_local_br"] & base["crise_global"]
base["normal"] = ~base["crise_local_br"] & ~base["crise_global"]
base["so_local_p90"] = base["crise_local_br"] & ~base["crise_global_p90"]

global_recente = base["crise_global"].astype(int).rolling(PREGOES_POS_GLOBAL, min_periods=1).max().shift(1).fillna(0)
base["so_local_sem_cauda"] = base["so_local"] & (global_recente == 0)

base["crise_local_fixa"] = False
posicoes = pd.Series(np.arange(len(base)), index=base.index)
for data_evento in EVENTOS_LOCAIS:
    inicio = posicoes[posicoes.index >= pd.Timestamp(data_evento)].iloc[0]
    base.iloc[inicio:inicio + PREGOES_EVENTO, base.columns.get_loc("crise_local_fixa")] = True

base["choque_local_br"] = base["r21_ibov"] < limiar_choque
base["so_local_choque"] = base["choque_local_br"] & ~base["crise_global"]

print(f"Percentil 90 do VIX na amostra: {limiar_vix_p90:.2f}")
print(f"Percentil 5 do retorno de 21 pregões do Ibovespa: {limiar_choque:.2f}%")

Percentil 90 do VIX na amostra: 27.76
Percentil 5 do retorno de 21 pregões do Ibovespa: -10.16%


## 8. Recorte da amostra e verificações

O recorte vai de `INICIO_AMOSTRA` a `FIM_AMOSTRA`. A primeira linha (16/09/2003) não tem retorno, porque é o primeiro valor do IMA-B. As verificações abaixo vão para o log; uma falha não interrompe o notebook, mas aparece como "FALHOU".

In [18]:
base = base.loc[INICIO_AMOSTRA:FIM_AMOSTRA].copy()
base.index.name = "data"
principais_ret = ["r_sp500", "r_treasury", "r_ibov", "r_ntnb", "r_usdbrl"]
corpo = base.iloc[1:]

verificacoes = []
def verificar(descricao, ok, detalhe=""):
    verificacoes.append({"verificação": descricao, "resultado": "ok" if ok else "FALHOU", "detalhe": detalhe})

verificar("Datas únicas e em ordem crescente", base.index.is_unique and base.index.is_monotonic_increasing)
ausentes = corpo[principais_ret + ["vix", "dd_ibov", "dd_sp500"]].isna().sum()
verificar("Sem ausentes nas séries principais (após a 1ª linha)", ausentes.sum() == 0,
          ", ".join(f"{k}: {v}" for k, v in ausentes.items() if v) or "nenhum")
niveis = ["sp500", "ibov", "imab", "usdbrl_ptax", "vix"]
verificar("Níveis positivos", bool((base[niveis] > 0).all().all()))

corr_ief = corpo[["r_treasury", "r_ief"]].corr().iloc[0, 1]
verificar("Correlação r_treasury x r_ief > 0,9", corr_ief > 0.9, f"{corr_ief:.3f}")
corr_aprox = corpo[["r_treasury", "r_treasury_aprox"]].corr().iloc[0, 1]
verificar("Correlação r_treasury x r_treasury_aprox > 0,99", corr_aprox > 0.99, f"{corr_aprox:.4f}")

# ANBIMA: retorno recalculado pelo número-índice x 'Variação Diária (%)' publicada (calendário da própria ANBIMA)
divergencias = []
for codigo, dados in anbima.items():
    calc = dados["Número Índice"].pct_change() * 100
    dif = (calc - dados["Variação Diária (%)"]).abs().loc[INICIO_AMOSTRA:FIM_AMOSTRA]
    for data_div, valor in dif[dif > 0.01].items():
        divergencias.append(f"{codigo} {data_div.date()} ({valor:.2f} p.p.)")
verificar("ANBIMA: número-índice x 'Variação Diária (%)' (dias divergentes > 0,01 p.p.)",
          True, "; ".join(divergencias) or "nenhum")

maiores = {}
for r in principais_ret:
    top = corpo[r].abs().nlargest(10)
    maiores[r] = [f"{d.date()} ({corpo.at[d, r]:+.2f}%)" for d in top.index]

contagem = pd.DataFrame({
    "dias": base[["crise_global", "crise_global_p90", "crise_local_br", "crise_eua", "crise_p1_eua",
                  "so_local", "so_global", "ambas", "normal", "so_local_p90", "so_local_sem_cauda",
                  "crise_local_fixa", "choque_local_br", "so_local_choque"]].sum()
})
contagem["% da amostra"] = (100 * contagem["dias"] / len(base)).round(1)
verificar("Regimes so_local/so_global/ambas/normal somam o total de dias",
          int(base[["so_local", "so_global", "ambas", "normal"]].sum().sum()) == len(base))

pd.DataFrame(verificacoes)

,verificação,resultado,detalhe
0,Datas únicas e em ordem crescente,ok,
1,Sem ausentes nas séries principais (após a 1ª ...,ok,nenhum
2,Níveis positivos,ok,
3,"Correlação r_treasury x r_ief > 0,9",ok,0.967
4,"Correlação r_treasury x r_treasury_aprox > 0,99",ok,1.0000
5,ANBIMA: número-índice x 'Variação Diária (%)' ...,ok,IDKAIPCA10A 2021-07-16 (0.20 p.p.); IRFM 2005-...
6,Regimes so_local/so_global/ambas/normal somam ...,ok,


In [19]:
contagem

,dias,% da amostra
crise_global,422,7.6
crise_global_p90,553,10.0
crise_local_br,729,13.2
crise_eua,314,5.7
crise_p1_eua,524,9.5
so_local,418,7.6
so_global,111,2.0
ambas,311,5.6
normal,4686,84.8
so_local_p90,378,6.8


## 9. Saídas

Salva a base, o dicionário de variáveis e o log. As colunas booleanas são gravadas como 0/1.

In [20]:
DICIONARIO = {
    "data": ("Data (dia útil comum a Brasil e EUA)", "aaaa-mm-dd", "—", "Interseção dos calendários das séries principais"),
    "sp500": ("S&P 500, fechamento", "pontos", "Yahoo Finance (^GSPC); S&P DJI", "Close"),
    "y_treasury10": ("Yield do Treasury de 10 anos, vencimento constante", "% a.a.", "Federal Reserve, H.15 (RIFLGFCY10_N.B)", "ND → ausente"),
    "vix": ("VIX, fechamento", "pontos", "Cboe (VIX_History.csv)", "CLOSE"),
    "ibov": ("Ibovespa, fechamento", "pontos", "Yahoo Finance (^BVSP); B3", "Close"),
    "imab": ("IMA-B, número-índice", "índice", "ANBIMA (IMAB)", "Número Índice"),
    "usdbrl_ptax": ("Câmbio PTAX venda", "R$/US$", "Banco Central, SGS 1", "texto → número"),
    "imab5mais": ("IMA-B 5+, número-índice", "índice", "ANBIMA (IMAB5MAIS)", "Número Índice"),
    "idka10": ("IDkA IPCA 10 anos, número-índice (desde 30/12/2005)", "índice", "ANBIMA (IDKAIPCA10A)", "Número Índice"),
    "irfm": ("IRF-M, número-índice", "índice", "ANBIMA (IRFM)", "Número Índice"),
    "ief": ("ETF IEF, preço ajustado por dividendos", "US$", "Yahoo Finance (IEF)", "Adj Close"),
    "y_tips10": ("Yield real do TIPS de 10 anos, vencimento constante", "% a.a.", "Federal Reserve, H.15 (RIFLGFCY10_XII_N.B)", "ND → ausente"),
    "usdbrl_fech": ("Câmbio USD/BRL de fechamento (desde dez. 2003)", "R$/US$", "Yahoo Finance (BRL=X)", f"Close; descartado se desvio da PTAX > {LIMIAR_DESVIO_BRL:.0%}"),
    "imab_duration_anos": ("Duration do IMA-B (desde 31/03/2005)", "anos", "ANBIMA", "Duration (d.u.) / 252"),
    "imab5mais_duration_anos": ("Duration do IMA-B 5+ (desde 31/03/2005)", "anos", "ANBIMA", "Duration (d.u.) / 252"),
    "irfm_duration_anos": ("Duration do IRF-M", "anos", "ANBIMA", "Duration (d.u.) / 252; 0 → ausente"),
    "dd_ibov": ("Drawdown do Ibovespa sobre o máximo de 252 pregões", "fração", "derivada", "calendário da B3"),
    "dd_sp500": ("Drawdown do S&P 500 sobre o máximo de 252 pregões", "fração", "derivada", "calendário da NYSE"),
    "r21_ibov": ("Log-retorno acumulado do Ibovespa em 21 pregões", "%", "derivada", "calendário da B3"),
    "r_sp500": ("Log-retorno diário do S&P 500", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_ibov": ("Log-retorno diário do Ibovespa", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_ntnb": ("Log-retorno diário do IMA-B (NTN-B, série principal)", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_ntnb5mais": ("Log-retorno diário do IMA-B 5+", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_idka10": ("Log-retorno diário do IDkA IPCA 10A", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_irfm": ("Log-retorno diário do IRF-M", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_ief": ("Log-retorno diário do IEF", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_usdbrl": ("Log-retorno diário da PTAX (positivo = real se desvaloriza)", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_usdbrl_fech": ("Log-retorno diário do BRL=X", "%", "derivada", "100·ln(P_t/P_t−1)"),
    "r_treasury": ("Retorno diário do Treasury de 10 anos", "%", "derivada", "título ao par reprecificado + carry; log"),
    "r_treasury_aprox": ("Retorno aproximado do Treasury (conferência)", "%", "derivada", "y_t−1·τ − D·Δy"),
    "r_tips": ("Retorno diário do TIPS de 10 anos", "%", "derivada", "título ao par reprecificado + carry; log"),
    "r_ibov_usd": ("Log-retorno do Ibovespa em dólar", "%", "derivada", "r_ibov − r_usdbrl"),
    "r_ntnb_usd": ("Log-retorno do IMA-B em dólar", "%", "derivada", "r_ntnb − r_usdbrl"),
    "dias_corridos": ("Dias corridos desde o dia útil comum anterior", "dias", "derivada", "diferença de datas"),
    "crise_global": ("VIX acima do limiar", "0/1", "derivada", f"vix > {LIMIAR_VIX}"),
    "crise_global_p90": ("VIX acima do percentil 90 da amostra", "0/1", "derivada", f"vix > {limiar_vix_p90:.2f}"),
    "crise_local_br": ("Ibovespa em drawdown", "0/1", "derivada", f"dd_ibov ≤ {LIMIAR_DRAWDOWN}"),
    "crise_eua": ("S&P 500 em drawdown", "0/1", "derivada", f"dd_sp500 ≤ {LIMIAR_DRAWDOWN}"),
    "crise_p1_eua": ("Crise de aversão a risco nos EUA (P1)", "0/1", "derivada", "crise_global ou crise_eua"),
    "so_local": ("Só crise local", "0/1", "derivada", "crise_local_br e não crise_global"),
    "so_global": ("Só crise global", "0/1", "derivada", "crise_global e não crise_local_br"),
    "ambas": ("Crise local e global", "0/1", "derivada", "crise_local_br e crise_global"),
    "normal": ("Sem crise local nem global", "0/1", "derivada", "nenhuma das duas"),
    "so_local_p90": ("Só crise local, com VIX p90", "0/1", "derivada", "crise_local_br e não crise_global_p90"),
    "so_local_sem_cauda": ("Só crise local, fora da cauda de crise global", "0/1", "derivada", f"so_local sem crise_global nos {PREGOES_POS_GLOBAL} pregões anteriores"),
    "crise_local_fixa": ("Janela após evento local da lista fixa (preliminar)", "0/1", "derivada", f"{PREGOES_EVENTO} pregões a partir de cada evento"),
    "choque_local_br": ("Choque de curto prazo no Ibovespa", "0/1", "derivada", f"r21_ibov < {limiar_choque:.2f}% (p5)"),
    "so_local_choque": ("Choque local sem crise global", "0/1", "derivada", "choque_local_br e não crise_global"),
}

colunas = [c for c in DICIONARIO if c != "data"]
faltando = set(base.columns) - set(colunas)
assert not faltando, f"Colunas sem descrição no dicionário: {faltando}"

saida = base[colunas].copy()
booleanas = saida.select_dtypes(bool).columns
saida[booleanas] = saida[booleanas].astype(int)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
saida.to_csv(PASTA_SAIDA / "base_diaria.csv", float_format="%.8g")

linhas = ["# Dicionário de variáveis – `dados/processados/base_diaria.csv`", "",
          "Gerado por `codigo/02_processamento.ipynb`. Retornos em % (log-retorno × 100). Indicadores de regime gravados como 0/1.", "",
          "| Coluna | Descrição | Unidade | Fonte | Transformação |", "|---|---|---|---|---|"]
linhas += [f"| `{c}` | {d} | {u} | {f} | {t} |" for c, (d, u, f, t) in DICIONARIO.items()]
(RAIZ / "dados" / "dicionario.md").write_text("\n".join(linhas) + "\n", encoding="utf-8")

datas = base.index.values.astype("datetime64[D]")
retornos_multiplos = int((np.busday_count(datas[:-1], datas[1:]) > 1).sum())
hash_manifesto = hashlib.sha256((PASTA_BRUTOS / "manifesto.csv").read_bytes()).hexdigest()
log = [
    "# Log do processamento", "",
    f"- Gerado em: {date.today().isoformat()}",
    f"- Brutos: `{PASTA_BRUTOS.relative_to(RAIZ)}` (SHA-256 do manifesto: `{hash_manifesto[:16]}…`)",
    f"- Amostra: {base.index.min().date()} a {base.index.max().date()}, {len(base)} dias úteis comuns",
    f"- Percentil 90 do VIX: {limiar_vix_p90:.2f}; percentil 5 de `r21_ibov`: {limiar_choque:.2f}%",
    f"- Retornos que cobrem mais de um dia útil (seg. a sex.) por causa de dia excluído: {retornos_multiplos}",
    "", "## Verificações", "", pd.DataFrame(verificacoes).to_markdown(index=False),
    "", "## Datas excluídas na amostra, por séries principais ausentes", "", tabela_excluidos.to_markdown(),
    "", f"## Observações descartadas do BRL=X (desvio da PTAX > {LIMIAR_DESVIO_BRL:.0%})", "",
    pd.DataFrame({"desvio": descartes_brl.round(4)}).to_markdown(),
    "", "## Dias por regime", "", contagem.to_markdown(),
    "", "## Eventos da lista fixa de crises locais (preliminar)", "",
    *[f"- {d}: {e}" for d, e in EVENTOS_LOCAIS.items()],
    "", "## Dez maiores retornos em módulo (conferir contra eventos conhecidos)", "",
]
for r, lista in maiores.items():
    log.append(f"- `{r}`: " + "; ".join(lista))
(PASTA_SAIDA / "log_processamento.md").write_text("\n".join(log) + "\n", encoding="utf-8")

print(f"base_diaria.csv: {saida.shape[0]} linhas x {saida.shape[1]} colunas")
print("dicionario.md e log_processamento.md atualizados")

base_diaria.csv: 5526 linhas x 47 colunas
dicionario.md e log_processamento.md atualizados
